In [ ]:
# 0. Mount Google Drive
# Colab will ask you to authorise access to your Drive.
from google.colab import drive
drive.mount("/content/drive")


# Building a small banking foundation model — end to end

This notebook asks a practical question before it asks a modelling question:

> **When does a transaction dataset actually justify sequence modelling and self-supervised pretraining?**

We first diagnose two public synthetic transaction datasets. We stop on **SynSEPA** when current-event shortcuts dominate the fraud label, then take the **IBM / Altman credit-card histories** forward into a small foundation-style experiment.

The notebook is designed for **Google Colab**. The setup is intentionally simple:

1. mount Google Drive;
2. create one project folder;
3. download the public datasets into Drive;
4. verify the files;
5. continue with diagnostics and modelling.

The data is stored in Drive so it remains available after the Colab runtime is reset.


In [ ]:
# 1. Project folders in Google Drive
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/banking-foundation-model")

DATA_DIR = PROJECT_ROOT / "data"
DATA_ROOT = DATA_DIR
SYNSEPA_DIR = DATA_DIR / "synsepa"
IBM_DIR = DATA_DIR / "ibm"

CACHE_ROOT = PROJECT_ROOT / "cache"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
ARTIFACTS_DIR = OUTPUT_ROOT / "artifacts"

for folder in [SYNSEPA_DIR, IBM_DIR, CACHE_ROOT, ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SYNSEPA_TX_PATH = SYNSEPA_DIR / "transactions.parquet"
SYNSEPA_ACCOUNTS_PATH = SYNSEPA_DIR / "accounts.parquet"

IBM_TX_CSV = IBM_DIR / "credit_card_transactions-ibm_v2.csv"
IBM_USERS_CSV = IBM_DIR / "sd254_users.csv"

print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)


## 2. Download the public datasets

### SynSEPA
Source: [EpiphanyTech/SynSEPA on Hugging Face](https://huggingface.co/datasets/EpiphanyTech/SynSEPA)

The dataset exposes two subsets:

- `transactions`
- `accounts`

The next cell downloads both and saves them as Parquet files in Google Drive.

### IBM / Altman credit-card transactions
Source: [Credit Card Transactions on Kaggle](https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions)

For this experiment we need:

- `credit_card_transactions-ibm_v2.csv`
- `sd254_users.csv`

The following Kaggle cell downloads those files directly into Google Drive.


In [ ]:
# 2a. Download SynSEPA from Hugging Face
!pip -q install datasets pyarrow

from datasets import load_dataset

if not SYNSEPA_TX_PATH.exists():
    print("Downloading SynSEPA transactions...")
    ds_transactions = load_dataset(
        "EpiphanyTech/SynSEPA",
        "transactions",
        split="train",
    )
    ds_transactions.to_parquet(str(SYNSEPA_TX_PATH))
    print("Saved:", SYNSEPA_TX_PATH)
else:
    print("Already present:", SYNSEPA_TX_PATH)

if not SYNSEPA_ACCOUNTS_PATH.exists():
    print("Downloading SynSEPA accounts...")
    ds_accounts = load_dataset(
        "EpiphanyTech/SynSEPA",
        "accounts",
        split="train",
    )
    ds_accounts.to_parquet(str(SYNSEPA_ACCOUNTS_PATH))
    print("Saved:", SYNSEPA_ACCOUNTS_PATH)
else:
    print("Already present:", SYNSEPA_ACCOUNTS_PATH)


In [ ]:
# 2b. Download the IBM / Altman files from Kaggle
!pip -q install kagglehub

import kagglehub

KAGGLE_DATASET = "ealtman2019/credit-card-transactions"

if not IBM_TX_CSV.exists():
    kagglehub.dataset_download(
        KAGGLE_DATASET,
        path="credit_card_transactions-ibm_v2.csv",
        output_dir=str(IBM_DIR),
    )
    print("Saved:", IBM_TX_CSV)
else:
    print("Already present:", IBM_TX_CSV)

if not IBM_USERS_CSV.exists():
    kagglehub.dataset_download(
        KAGGLE_DATASET,
        path="sd254_users.csv",
        output_dir=str(IBM_DIR),
    )
    print("Saved:", IBM_USERS_CSV)
else:
    print("Already present:", IBM_USERS_CSV)


In [ ]:
# 3. Verify the data and load SynSEPA
import pandas as pd

required_files = {
    "SynSEPA transactions": SYNSEPA_TX_PATH,
    "SynSEPA accounts": SYNSEPA_ACCOUNTS_PATH,
    "IBM transactions": IBM_TX_CSV,
    "IBM users": IBM_USERS_CSV,
}

for name, path in required_files.items():
    print(f"{'✓' if path.exists() else '✗'} {name}: {path}")

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Some required data files are missing. Rerun the download cells above:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

synsepa_tx = pd.read_parquet(SYNSEPA_TX_PATH)
synsepa_accounts = pd.read_parquet(SYNSEPA_ACCOUNTS_PATH)
synsepa_tx["timestamp"] = pd.to_datetime(synsepa_tx["timestamp"])

print()
print(f"SynSEPA transactions: {len(synsepa_tx):,}")
print(f"SynSEPA accounts in transactions: {synsepa_tx['account_id'].nunique():,}")
print(f"SynSEPA account records: {len(synsepa_accounts):,}")
print(
    "SynSEPA date range:",
    synsepa_tx["timestamp"].min(),
    "→",
    synsepa_tx["timestamp"].max(),
)

print()
print(f"IBM transaction file: {IBM_TX_CSV.stat().st_size / 1024**3:.2f} GB")
print(f"IBM users file: {IBM_USERS_CSV.stat().st_size / 1024**2:.2f} MB")

display(synsepa_tx.head())
display(synsepa_accounts.head())

print("✓ Data setup complete. Continue to Part I.")


In [ ]:
# 4. Dependencies, reproducibility and run controls.
import sys, subprocess, importlib.util, math, time, random, json, pickle, copy, gc
from dataclasses import dataclass

REQ = {
    "pyarrow": "pyarrow>=12",
    "lightgbm": "lightgbm>=4.0",
    "sklearn": "scikit-learn>=1.3",
    "torch": "torch",
}
missing_pkgs = [pkg for mod, pkg in REQ.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
SEED_SPLIT = 1001
SEED_BOOT = 2002
SEED_MODEL = 3003
SEED_RANDOM_CONTROL = 4242

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# FAST_MODE is the public default. Set False for more pretraining windows/steps.
FAST_MODE = True
FORCE_REBUILD_CACHE = False
RUN_REPRESENTATION_ABLATIONS = True

CFG = dict(
    max_events=64,
    windows_per_train_user=20 if FAST_MODE else 40,
    windows_per_val_user=12 if FAST_MODE else 20,
    min_pretrain_events=8,
    numeric_bins=32,
    d_model=128,
    n_heads=4,
    d_ff=512,
    profile_layers=1,
    event_layers=1,
    history_layers=2,
    dropout=0.10,
    token_mask_rate=0.15,
    event_mask_rate=0.05,
    field_mask_rate=0.05,
    unk_corruption_rate=0.10,
    pretrain_batch_size=64,
    pretrain_lr=3e-4,
    pretrain_weight_decay=0.01,
    max_pretrain_steps=5_000 if FAST_MODE else 10_000,
    warmup_steps=250,
    validate_every=500,
    validation_batches=40 if FAST_MODE else 80,
    checkpoint_every=1_000,
    downstream_batch_size=128,
    scratch_epochs=3 if FAST_MODE else 5,
    finetune_epochs=3 if FAST_MODE else 5,
    scratch_lr=2e-4,
    finetune_lr=5e-5,
    probe_max_iter=400,
    n_boot=200 if FAST_MODE else 500,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("lightgbm:", lgb.__version__)
print("device:", DEVICE)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


# Part I — Should we use a sequence model at all?

A transaction table can be huge and still be a poor foundation-model test. We use three diagnostics throughout:

1. **History depth** — are there repeated observations over a meaningful period, and do enough entities have long sequences?
2. **Shortcut risk** — can current-event fields or a simple rule already explain most of the target?
3. **Incremental role for history** — does adding historical context improve over current-event-only information?

These are diagnostics rather than universal hard thresholds. The purpose is to force the modelling decision to be justified by the data.


In [ ]:
# 1. Reusable diagnostic and plotting helpers.
def history_summary(df, entity_col, timestamp_col, sequence_thresholds=(16, 32, 64)):
    x = df[[entity_col, timestamp_col]].copy()
    x[timestamp_col] = pd.to_datetime(x[timestamp_col], errors="coerce")
    x = x.dropna(subset=[entity_col, timestamp_col])
    g = x.groupby(entity_col)[timestamp_col].agg(["count", "min", "max"])
    g["span_days"] = (g["max"] - g["min"]).dt.total_seconds() / 86400.0
    out = {
        "entities": int(len(g)),
        "events": int(len(x)),
        "events_p25": float(g["count"].quantile(.25)),
        "events_median": float(g["count"].median()),
        "events_p90": float(g["count"].quantile(.90)),
        "span_days_median": float(g["span_days"].median()),
    }
    for n in sequence_thresholds:
        out[f"pct_entities_ge_{n}_events"] = float((g["count"] >= n).mean())
    return out, g


def savefig(name):
    path = ARTIFACTS_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("saved ->", path)
    return path


def plot_history_distribution(entity_stats, title, filename):
    counts = entity_stats["count"].clip(upper=entity_stats["count"].quantile(.99))
    plt.figure(figsize=(8.6, 4.7))
    plt.hist(counts, bins=40)
    plt.axvline(64, linestyle="--", linewidth=1.5, label="64-event history")
    plt.xlabel("Transactions per entity (clipped at p99)")
    plt.ylabel("Entities")
    plt.title(title)
    plt.legend(frameon=False)
    savefig(filename)
    plt.show()


def metric_bar(table, x, y, title, filename, ylabel=None):
    plt.figure(figsize=(9, 4.8))
    plt.bar(table[x].astype(str), table[y].astype(float))
    plt.xticks(rotation=20, ha="right")
    plt.ylabel(ylabel or y)
    plt.title(title)
    savefig(filename)
    plt.show()


## 2. Dataset 1 — SynSEPA

SynSEPA is useful precisely because it demonstrates why “many transactions” is not enough. We inspect the histories, then test whether the fraud label is dominated by information available on the current payment.


In [ ]:
# 2a. Use the SynSEPA transactions loaded during setup.
SYN = synsepa_tx[
    [
        "transaction_id", "account_id", "persona", "timestamp",
        "beneficiary_country", "country_type", "amount", "remittance_category",
        "hour_of_day", "day_of_week", "is_weekend", "time_since_last_txn",
        "is_new_beneficiary", "is_fraud", "fraud_type",
    ]
].copy()

SYN["timestamp"] = pd.to_datetime(SYN["timestamp"], errors="coerce")
SYN["is_fraud"] = pd.to_numeric(SYN["is_fraud"], errors="coerce").fillna(0).astype(np.int8)
SYN["is_new_beneficiary"] = pd.to_numeric(SYN["is_new_beneficiary"], errors="coerce").fillna(0).astype(np.int8)

syn_summary, syn_entity_stats = history_summary(SYN, "account_id", "timestamp")
display(pd.DataFrame([syn_summary]).T.rename(columns={0: "SynSEPA"}))
plot_history_distribution(
    syn_entity_stats,
    "SynSEPA — history depth",
    "01_synsepa_history_depth.png",
)


In [ ]:
# 2b. Shortcut diagnostic: fraud rate by current transaction type and new-beneficiary flag.
shortcut = (
    SYN.groupby(["country_type", "is_new_beneficiary"], observed=True)["is_fraud"]
       .agg(["count", "mean", "sum"])
       .rename(columns={"mean": "fraud_rate", "sum": "fraud_cases"})
       .reset_index()
       .sort_values("fraud_rate", ascending=False)
)
display(shortcut)

pivot = shortcut.pivot(index="country_type", columns="is_new_beneficiary", values="fraud_rate").fillna(0)
plt.figure(figsize=(7.5, 4.8))
im = plt.imshow(pivot.to_numpy(), aspect="auto")
plt.colorbar(im, label="Fraud rate")
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xticks(range(len(pivot.columns)), [f"new={c}" for c in pivot.columns])
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        plt.text(j, i, f"{pivot.iloc[i,j]:.1%}", ha="center", va="center")
plt.title("SynSEPA — current-event shortcut diagnostic")
savefig("02_synsepa_shortcut_heatmap.png")
plt.show()

print("Overall fraud prevalence:", f"{SYN['is_fraud'].mean():.3%}")


In [ ]:
# 2c. Current-event-only vs current + basic-history diagnostic.
# This is deliberately a lightweight diagnostic, not the final fraud benchmark.
# We sample rows to keep the check fast and split by account so the same account never appears in train and test.

rng = np.random.default_rng(SEED)
accounts = SYN["account_id"].drop_duplicates().to_numpy()
rng.shuffle(accounts)
cut = int(.8 * len(accounts))
train_accounts, test_accounts = set(accounts[:cut]), set(accounts[cut:])

sample_n = min(350_000, len(SYN))
SYN_DIAG = SYN.sample(sample_n, random_state=SEED) if len(SYN) > sample_n else SYN.copy()
tr = SYN_DIAG["account_id"].isin(train_accounts).to_numpy()
te = SYN_DIAG["account_id"].isin(test_accounts).to_numpy()

current_features = ["amount", "country_type", "is_new_beneficiary", "hour_of_day", "is_weekend", "persona"]
history_features = current_features + ["time_since_last_txn"]


def quick_lgbm_ap(features):
    X = SYN_DIAG[features].copy()
    cats = [c for c in features if X[c].dtype == "object" or str(X[c].dtype).startswith("string")]
    for c in cats:
        X[c] = X[c].astype("category")
    model = lgb.LGBMClassifier(
        objective="binary", n_estimators=250, learning_rate=.06, num_leaves=31,
        min_child_samples=100, colsample_bytree=.8, subsample=.8,
        random_state=SEED_MODEL, verbosity=-1,
    )
    model.fit(X.loc[tr], SYN_DIAG.loc[tr, "is_fraud"], categorical_feature=cats,
              callbacks=[lgb.log_evaluation(0)])
    score = model.predict_proba(X.loc[te])[:,1]
    return average_precision_score(SYN_DIAG.loc[te, "is_fraud"], score)

syn_diag_scores = pd.DataFrame([
    {"model": "Current event only", "average_precision": quick_lgbm_ap(current_features)},
    {"model": "Current + supplied history field", "average_precision": quick_lgbm_ap(history_features)},
])
display(syn_diag_scores)
metric_bar(syn_diag_scores, "model", "average_precision",
           "SynSEPA — does basic history add much beyond the current event?",
           "03_synsepa_current_vs_history.png", "Average precision")


### SynSEPA decision

The useful lesson is not “SynSEPA is a bad dataset.” It is that **this sampled fraud target is a weak test of the foundation-model hypothesis** when current-payment attributes already carry unusually strong label information.

We therefore stop here rather than training a large sequence model just because the table contains transaction histories.


## 3. Dataset 2 — IBM / Altman credit-card histories

For IBM / Altman we take a different route: rather than using the fraud label as the downstream target, we define **forward-looking behavioural questions** where past history plausibly matters.

The public benchmark below uses transactions from 2014 onward and monthly evaluation points through the end of 2019. The raw-to-panel logic is included here so the public repository is reproducible from the downloadable CSVs.


In [ ]:
# 3a. Prepare IBM transaction history from the raw CSV (cached after first run).
IBM_TX_PARQUET = CACHE_ROOT / "ibm_transactions_2014plus.parquet"
IBM_USERS_PARQUET = CACHE_ROOT / "ibm_users.parquet"


def money_to_float(s):
    return pd.to_numeric(s.astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

if FORCE_REBUILD_CACHE or not IBM_TX_PARQUET.exists():
    usecols = [
        "User", "Card", "Year", "Month", "Day", "Time", "Amount", "Use Chip",
        "Merchant City", "Merchant State", "MCC", "Is Fraud?",
    ]
    parts = []
    for chunk_i, x in enumerate(pd.read_csv(IBM_TX_CSV, usecols=usecols, chunksize=1_000_000, low_memory=False)):
        x = x.loc[pd.to_numeric(x["Year"], errors="coerce") >= 2014].copy()
        if x.empty: continue
        date = pd.to_datetime(dict(
            year=pd.to_numeric(x["Year"], errors="coerce"),
            month=pd.to_numeric(x["Month"], errors="coerce"),
            day=pd.to_numeric(x["Day"], errors="coerce"),
        ), errors="coerce")
        td = pd.to_timedelta(x["Time"].astype(str) + ":00", errors="coerce")
        merchant_city = x["Merchant City"].astype("string").fillna("").str.strip()
        merchant_state = x["Merchant State"].astype("string").fillna("").str.strip()
        is_online = merchant_city.str.upper().eq("ONLINE")
        state_missing = merchant_state.eq("")
        # In this public benchmark, a non-online payment with no US-state value is treated as foreign/unknown geography.
        is_foreign = (~is_online) & state_missing
        y = pd.DataFrame({
            "User": pd.to_numeric(x["User"], errors="coerce").astype("Int64"),
            "timestamp": date + td,
            "amount": money_to_float(x["Amount"]).astype("float32"),
            "is_fraud": x["Is Fraud?"].astype("string").str.strip().str.lower().eq("yes").astype("int8"),
            "mcc": pd.to_numeric(x["MCC"], errors="coerce").fillna(-1).astype("int32").astype("string"),
            "use_chip": x["Use Chip"].astype("string").fillna("__MISSING__"),
            "merchant_state": merchant_state.replace("", "__MISSING__"),
            "is_online": is_online.astype("int8").astype("string"),
            "is_foreign": is_foreign.astype("int8").astype("string"),
            "state_missing": state_missing.astype("int8").astype("string"),
        }).dropna(subset=["User", "timestamp"])
        y["User"] = y["User"].astype("int32")
        parts.append(y)
        if chunk_i % 5 == 0: print("processed raw chunk", chunk_i)
    TX = pd.concat(parts, ignore_index=True)
    TX = TX.sort_values(["User", "timestamp"], kind="stable").reset_index(drop=True)
    TX.to_parquet(IBM_TX_PARQUET, index=False)
    del parts
    gc.collect()
    print("saved ->", IBM_TX_PARQUET)
else:
    TX = pd.read_parquet(IBM_TX_PARQUET)

if FORCE_REBUILD_CACHE or not IBM_USERS_PARQUET.exists():
    USERS = pd.read_csv(IBM_USERS_CSV, low_memory=False)
    if "User" not in USERS.columns:
        USERS.insert(0, "User", np.arange(len(USERS), dtype=np.int32))
    USERS["User"] = pd.to_numeric(USERS["User"], errors="coerce").astype("int32")
    USERS.to_parquet(IBM_USERS_PARQUET, index=False)
else:
    USERS = pd.read_parquet(IBM_USERS_PARQUET)

TX["timestamp"] = pd.to_datetime(TX["timestamp"])
print(f"IBM transactions: {len(TX):,}")
print("IBM users in transactions:", TX["User"].nunique())
print("transaction range:", TX["timestamp"].min(), "->", TX["timestamp"].max())


In [ ]:
# 3b. IBM history-depth diagnostics.
ibm_summary, ibm_entity_stats = history_summary(TX, "User", "timestamp")
display(pd.DataFrame([ibm_summary]).T.rename(columns={0: "IBM / Altman"}))
plot_history_distribution(ibm_entity_stats, "IBM / Altman — history depth", "04_ibm_history_depth.png")

comparison_cols = [
    "entities", "events", "events_median", "events_p90", "span_days_median",
    "pct_entities_ge_64_events",
]
dataset_diag = pd.DataFrame({
    "SynSEPA": {k: syn_summary[k] for k in comparison_cols},
    "IBM / Altman": {k: ibm_summary[k] for k in comparison_cols},
}).T

display(dataset_diag)

plt.figure(figsize=(8.5, 4.5))
plot_df = pd.DataFrame({
    "dataset": ["SynSEPA", "IBM / Altman"],
    "median_events": [syn_summary["events_median"], ibm_summary["events_median"]],
})
plt.bar(plot_df["dataset"], plot_df["median_events"])
plt.ylabel("Median transactions per entity")
plt.title("History depth differs materially across candidate datasets")
savefig("05_dataset_history_comparison.png")
plt.show()


### IBM decision

The IBM histories are taken forward because they support repeated observations over long periods and make it possible to ask questions about **what the customer does next**, rather than merely predicting a label attached to the current transaction.

The two tasks will be:

- **Task 1 — future category use:** will the user transact in each of five merchant-category groups in the next 30 days?
- **Task 2 — future activity tier:** will next-30-day transaction activity be LOW, MEDIUM or HIGH?


# Part II — Define the public IBM benchmark

The original development notebook used a cached panel whose raw construction script was not preserved. For the public repository, the benchmark is therefore defined transparently below.

This is important: the code below is **the definition of the public benchmark**, not a claim that it is byte-for-byte identical to the missing development preprocessing.

Five interpretable MCC groups are used. They are analyst-defined coarse groups for this experiment, not an official taxonomy:

- `personal_services`: MCC 7200–7299
- `entertainment_rec`: MCC 7800–7999
- `transport_travel`: MCC 3000–3999 or 4000–4799
- `financial_services`: MCC 6000–6999
- `government_educ`: MCC 8200–8299 or 9000–9999

Everything else is `OTHER` and is excluded from the headline multilabel task.


In [ ]:
# 4. Build the monthly forward-looking panel from raw transaction histories.
PANEL_PARQUET = CACHE_ROOT / "ibm_public_monthly_panel.parquet"
TARGET_GROUPS = [
    "personal_services", "entertainment_rec", "transport_travel",
    "financial_services", "government_educ",
]


def mcc_group(mcc):
    m = pd.to_numeric(mcc, errors="coerce").fillna(-1).astype(int)
    out = np.full(len(m), "OTHER", dtype=object)
    out[(m >= 7200) & (m <= 7299)] = "personal_services"
    out[(m >= 7800) & (m <= 7999)] = "entertainment_rec"
    out[((m >= 3000) & (m <= 3999)) | ((m >= 4000) & (m <= 4799))] = "transport_travel"
    out[(m >= 6000) & (m <= 6999)] = "financial_services"
    out[((m >= 8200) & (m <= 8299)) | ((m >= 9000) & (m <= 9999))] = "government_educ"
    return out


def range_sum(cs, left, right):
    # cs has a leading zero; left/right are event indices, right exclusive.
    return cs[right] - cs[left]

if FORCE_REBUILD_CACHE or not PANEL_PARQUET.exists():
    tx_group = mcc_group(TX["mcc"])
    TX["mcc_group"] = pd.Categorical(tx_group, categories=TARGET_GROUPS + ["OTHER"])
    eval_points = pd.date_range("2014-01-31", "2019-12-31", freq="ME") + pd.Timedelta(hours=23, minutes=59, seconds=59)
    eval_s_all = eval_points.to_numpy("datetime64[s]").astype("int64")
    sec30, sec90 = 30*86400, 90*86400
    rows = []

    for ui, (user, g) in enumerate(TX.groupby("User", sort=False, observed=True)):
        t = g["timestamp"].to_numpy("datetime64[s]").astype("int64")
        amt = g["amount"].fillna(0).to_numpy(np.float64)
        online = pd.to_numeric(g["is_online"], errors="coerce").fillna(0).to_numpy(np.float64)
        foreign = pd.to_numeric(g["is_foreign"], errors="coerce").fillna(0).to_numpy(np.float64)
        gg = g["mcc_group"].astype("string").to_numpy()

        cs_amt = np.r_[0.0, np.cumsum(amt)]
        cs_online = np.r_[0.0, np.cumsum(online)]
        cs_foreign = np.r_[0.0, np.cumsum(foreign)]
        cs_group = {name: np.r_[0, np.cumsum(gg == name)] for name in TARGET_GROUPS}

        right = np.searchsorted(t, eval_s_all, side="right")
        left30 = np.searchsorted(t, eval_s_all - sec30, side="right")
        left90 = np.searchsorted(t, eval_s_all - sec90, side="right")
        fut = np.searchsorted(t, eval_s_all + sec30, side="right")

        # Require at least 64 historical events. The global eval range stops in 2019,
        # leaving 2020 transactions available for the final 30-day horizon where present.
        valid = right >= CFG["max_events"]
        for j in np.flatnonzero(valid):
            n30 = int(right[j] - left30[j])
            n90 = int(right[j] - left90[j])
            row = {
                "User": int(user),
                "eval_point": eval_points[j],
                "eval_month": int(eval_points[j].month),
                "p30_tx_count": n30,
                "p90_tx_count": n90,
                "p30_amount_sum": float(range_sum(cs_amt, left30[j], right[j])),
                "p90_amount_sum": float(range_sum(cs_amt, left90[j], right[j])),
                "p30_amount_mean": float(range_sum(cs_amt, left30[j], right[j]) / max(n30, 1)),
                "p90_amount_mean": float(range_sum(cs_amt, left90[j], right[j]) / max(n90, 1)),
                "p30_online_rate": float(range_sum(cs_online, left30[j], right[j]) / max(n30, 1)),
                "p90_online_rate": float(range_sum(cs_online, left90[j], right[j]) / max(n90, 1)),
                "p30_foreign_rate": float(range_sum(cs_foreign, left30[j], right[j]) / max(n30, 1)),
                "p90_foreign_rate": float(range_sum(cs_foreign, left90[j], right[j]) / max(n90, 1)),
                "future_30d_tx_count": int(fut[j] - right[j]),
            }
            for name in TARGET_GROUPS:
                cs = cs_group[name]
                c30 = int(range_sum(cs, left30[j], right[j]))
                c90 = int(range_sum(cs, left90[j], right[j]))
                cf = int(range_sum(cs, right[j], fut[j]))
                row[f"p30_{name}_count"] = c30
                row[f"p90_{name}_count"] = c90
                row[f"prev_{name}"] = int(c30 > 0)
                row[f"y_{name}"] = int(cf > 0)
            rows.append(row)
        if ui % 250 == 0: print("panel users processed:", ui)

    PANEL = pd.DataFrame(rows).sort_values(["eval_point", "User"]).reset_index(drop=True)
    PANEL.to_parquet(PANEL_PARQUET, index=False)
    print("saved ->", PANEL_PARQUET)
else:
    PANEL = pd.read_parquet(PANEL_PARQUET)

PANEL["eval_point"] = pd.to_datetime(PANEL["eval_point"])
print(f"Panel: {len(PANEL):,} rows; {PANEL['User'].nunique():,} users")
print("Panel range:", PANEL["eval_point"].min(), "->", PANEL["eval_point"].max())
display(PANEL.head())


In [ ]:
# 5. Entity-disjoint train / validation / test split.
@dataclass
class SplitBundle:
    train: np.ndarray
    val: np.ndarray
    test: np.ndarray
    seed: int = SEED_SPLIT


def make_entity_split(df, entity_col="User", stratify_col="is_fraud", fracs=(.70,.10,.20), seed=SEED_SPLIT):
    ents = df[entity_col].drop_duplicates().to_numpy()
    strat = (df.groupby(entity_col)[stratify_col].max() > 0).astype(int).reindex(ents).fillna(0).to_numpy()
    rng = np.random.default_rng(seed)
    parts = {"train": [], "val": [], "test": []}
    for grp in np.unique(strat):
        sel = ents[strat == grp]
        sel = sel[rng.permutation(len(sel))]
        n = len(sel); ntr = int(round(fracs[0]*n)); nva = int(round(fracs[1]*n))
        parts["train"].append(sel[:ntr]); parts["val"].append(sel[ntr:ntr+nva]); parts["test"].append(sel[ntr+nva:])
    tr, va, te = [np.concatenate(parts[k]) for k in ("train","val","test")]
    assert not (set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
    return SplitBundle(tr,va,te,seed)

SPLIT = make_entity_split(TX)
TRAIN_USERS, VAL_USERS, TEST_USERS = map(set, [SPLIT.train.tolist(), SPLIT.val.tolist(), SPLIT.test.tolist()])
panel_user = PANEL["User"].to_numpy()
P_TR = np.isin(panel_user, SPLIT.train)
P_VA = np.isin(panel_user, SPLIT.val)
P_TE = np.isin(panel_user, SPLIT.test)

split_table = pd.DataFrame({
    "split":["train","validation","test"],
    "users":[len(SPLIT.train),len(SPLIT.val),len(SPLIT.test)],
    "panel_rows":[int(P_TR.sum()),int(P_VA.sum()),int(P_TE.sum())],
})
display(split_table)

base_rates = pd.DataFrame({
    "category": TARGET_GROUPS,
    "train_rate": [PANEL.loc[P_TR, f"y_{g}"].mean() for g in TARGET_GROUPS],
    "test_rate": [PANEL.loc[P_TE, f"y_{g}"].mean() for g in TARGET_GROUPS],
})
display(base_rates)
metric_bar(base_rates, "category", "test_rate", "Task 1 — next-30-day target prevalence", "06_task1_target_prevalence.png", "Positive rate")


# Part III — Pretrain a reusable behavioural representation

We now train a small hierarchical Transformer over customer histories. The sequence inputs deliberately exclude the fraud label and high-cardinality merchant identifiers.

The self-supervised objective masks parts of historical events and asks the model to reconstruct them from the surrounding customer context. We compare reconstruction accuracy with two cheap references:

- **unigram** — predict the most common token for that field;
- **recency copy** — copy the same field from the previous event.


In [ ]:
# 6. Freeze the public model input fields.
SAFE_PROFILE_NUMERIC = ["Birth Year"]
SAFE_PROFILE_CATEGORICAL = ["Birth Month", "Gender"]
EVENT_NUMERIC = ["amount"]
EVENT_CATEGORICAL = ["mcc", "use_chip", "merchant_state", "is_online", "is_foreign", "state_missing"]

for c in EVENT_CATEGORICAL:
    TX[c] = TX[c].astype("string")

event_numeric = [c for c in EVENT_NUMERIC if c in TX.columns]
event_categorical = [c for c in EVENT_CATEGORICAL if c in TX.columns]
profile_numeric = [c for c in SAFE_PROFILE_NUMERIC if c in USERS.columns]
profile_categorical = [c for c in SAFE_PROFILE_CATEGORICAL if c in USERS.columns]

print("event numeric:", event_numeric)
print("event categorical:", event_categorical)
print("profile numeric:", profile_numeric)
print("profile categorical:", profile_categorical)


In [ ]:
# 6. Structured tokeniser

class StructuredTokenizer:
    PAD = "[PAD]"
    MASK = "[MASK]"
    UNK = "[UNK]"
    EVT = "[EVT]"
    USR = "[USR]"

    def __init__(self, numeric_bins=32):
        self.numeric_bins = numeric_bins
        self.token_to_id = {}
        self.id_to_token = []
        self.numeric_edges = {}
        self.cat_maps = {}
        self.event_fields = []
        self.profile_fields = []
        for tok in [self.PAD, self.MASK, self.UNK, self.EVT, self.USR]:
            self._add(tok)

    def _add(self, token):
        if token not in self.token_to_id:
            self.token_to_id[token] = len(self.id_to_token)
            self.id_to_token.append(token)
        return self.token_to_id[token]

    @property
    def pad_id(self): return self.token_to_id[self.PAD]
    @property
    def mask_id(self): return self.token_to_id[self.MASK]
    @property
    def unk_id(self): return self.token_to_id[self.UNK]
    @property
    def evt_id(self): return self.token_to_id[self.EVT]
    @property
    def usr_id(self): return self.token_to_id[self.USR]
    @property
    def vocab_size(self): return len(self.id_to_token)

    def fit_numeric(self, field, s):
        x = pd.to_numeric(s, errors="coerce").to_numpy(np.float64)
        x = x[np.isfinite(x)]
        if len(x) > 1_000_000:
            rng = np.random.default_rng(SEED)
            x = rng.choice(x, size=1_000_000, replace=False)
        qs = np.quantile(x, np.linspace(0, 1, self.numeric_bins + 1))
        qs = np.unique(qs)
        if len(qs) < 3:
            qs = np.array([-np.inf, np.inf])
        self.numeric_edges[field] = qs
        self._add(f"KEY::{field}")
        self._add(f"VAL::{field}::__MISSING__")
        for i in range(max(1, len(qs) - 1)):
            self._add(f"VAL::{field}::BIN_{i}")

    def fit_categorical(self, field, s, max_levels=1000):
        vals = s.astype("string").fillna("__MISSING__")
        vc = vals.value_counts(dropna=False)
        levels = vc.index[:max_levels].astype(str).tolist()
        self._add(f"KEY::{field}")
        mapping = {}
        for v in levels:
            mapping[v] = self._add(f"VAL::{field}::{v}")
        self.cat_maps[field] = mapping

    def fit(self, tx_train, users_train,
            event_numeric, event_categorical,
            profile_numeric, profile_categorical):
        self.event_fields = list(event_numeric) + list(event_categorical)
        self.profile_fields = list(profile_numeric) + list(profile_categorical)

        for f in event_numeric:
            self.fit_numeric(f, tx_train[f])
        for f in event_categorical:
            self.fit_categorical(f, tx_train[f])

        for f in profile_numeric:
            self.fit_numeric(f, users_train[f])
        for f in profile_categorical:
            self.fit_categorical(f, users_train[f], max_levels=500)

        self.event_key_ids = np.array(
            [self.token_to_id[f"KEY::{f}"] for f in self.event_fields], dtype=np.int64)
        self.profile_key_ids = np.array(
            [self.token_to_id[f"KEY::{f}"] for f in self.profile_fields], dtype=np.int64)
        return self

    def _encode_numeric(self, field, s):
        x = pd.to_numeric(s, errors="coerce").to_numpy(np.float64)
        out = np.full(len(x), self.token_to_id[f"VAL::{field}::__MISSING__"], dtype=np.int32)
        ok = np.isfinite(x)
        edges = self.numeric_edges[field]
        if len(edges) <= 2:
            bins = np.zeros(ok.sum(), dtype=np.int32)
        else:
            bins = np.searchsorted(edges[1:-1], x[ok], side="right").astype(np.int32)
        ids = np.array(
            [self.token_to_id[f"VAL::{field}::BIN_{i}"]
             for i in range(max(1, len(edges)-1))], dtype=np.int32)
        out[ok] = ids[np.clip(bins, 0, len(ids)-1)]
        return out

    def _encode_cat(self, field, s):
        mapping = self.cat_maps[field]
        return (s.astype("string").fillna("__MISSING__").astype(str)
                .map(mapping).fillna(self.unk_id).astype(np.int32).to_numpy())

    def encode_events(self, df):
        cols = []
        for f in self.event_fields:
            if f in self.numeric_edges:
                cols.append(self._encode_numeric(f, df[f]))
            else:
                cols.append(self._encode_cat(f, df[f]))
        return np.column_stack(cols).astype(np.int32)

    def encode_profiles(self, df):
        if not self.profile_fields:
            return np.zeros((len(df), 0), dtype=np.int32)
        cols = []
        for f in self.profile_fields:
            if f in self.numeric_edges:
                cols.append(self._encode_numeric(f, df[f]))
            else:
                cols.append(self._encode_cat(f, df[f]))
        return np.column_stack(cols).astype(np.int32)

train_user_set = set(SPLIT.train.tolist())
tx_train_mask = TX["User"].isin(train_user_set)
users_train = USERS[USERS["User"].isin(train_user_set)].copy()

TOKENIZER_PATH = CACHE_ROOT / "tokenizer_public.pkl"

if TOKENIZER_PATH.exists() and not FORCE_REBUILD_CACHE:
    with open(TOKENIZER_PATH, "rb") as f:
        TOK = pickle.load(f)
    print("Loaded tokenizer:", TOKENIZER_PATH)
else:
    TOK = StructuredTokenizer(CFG["numeric_bins"]).fit(
        TX.loc[tx_train_mask],
        users_train,
        event_numeric, event_categorical,
        profile_numeric, profile_categorical,
    )
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(TOK, f)
    print("Saved tokenizer:", TOKENIZER_PATH)

print("vocab size:", TOK.vocab_size)
print("event fields:", TOK.event_fields)
print("profile fields:", TOK.profile_fields)


In [ ]:
# 7. Compact encoded event store

ENCODED_CACHE = CACHE_ROOT / "event_store_public.npz"

def calendar_features(ts):
    ts = pd.to_datetime(ts)
    hour = ts.dt.hour.to_numpy(np.float32)
    dow = ts.dt.dayofweek.to_numpy(np.float32)
    month = ts.dt.month.to_numpy(np.float32)
    return np.column_stack([
        np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7),
        np.sin(2*np.pi*(month-1)/12), np.cos(2*np.pi*(month-1)/12),
    ]).astype(np.float32)

if ENCODED_CACHE.exists() and not FORCE_REBUILD_CACHE:
    z = np.load(ENCODED_CACHE, mmap_mode=None)
    EV_VALUES = z["values"]
    EV_TIMES = z["times"]
    EV_CAL = z["cal"]
    EV_USERS = z["users"]
    print("Loaded encoded event cache:", ENCODED_CACHE)
else:
    t0 = time.time()
    EV_VALUES = TOK.encode_events(TX)
    EV_TIMES = TX["timestamp"].to_numpy("datetime64[s]").astype("int64")
    EV_CAL = calendar_features(TX["timestamp"])
    EV_USERS = TX["User"].to_numpy(np.int32)
    np.savez_compressed(
        ENCODED_CACHE,
        values=EV_VALUES,
        times=EV_TIMES,
        cal=EV_CAL,
        users=EV_USERS,
    )
    print(f"Encoded and cached in {time.time()-t0:.1f}s -> {ENCODED_CACHE}")

# Contiguous user bounds because TX is sorted by User,timestamp.
uniq_users, starts, counts = np.unique(EV_USERS, return_index=True, return_counts=True)
USER_BOUNDS = {
    int(u): (int(s), int(s+c))
    for u, s, c in zip(uniq_users, starts, counts)
}

print("encoded event matrix:", EV_VALUES.shape, EV_VALUES.dtype)
print("calendar features:", EV_CAL.shape)
print("users in store:", len(USER_BOUNDS))


In [ ]:
# 8. Encode profile state per user

PROFILE_VALUES = TOK.encode_profiles(USERS)
PROFILE_BY_USER = {
    int(u): PROFILE_VALUES[i]
    for i, u in enumerate(USERS["User"].to_numpy())
}

def get_profile_values(user):
    return PROFILE_BY_USER.get(
        int(user),
        np.full(len(TOK.profile_fields), TOK.unk_id, dtype=np.int32)
    )

print("profile token width:", len(TOK.profile_fields))
if len(TOK.profile_fields):
    sample_user = int(SPLIT.train[0])
    print("sample user:", sample_user, get_profile_values(sample_user))


In [ ]:
# 9. Event-store helpers and sampled pre-training records

MAX_EVENTS = CFG["max_events"]
N_FIELDS = len(TOK.event_fields)
TIME_DIM = EV_CAL.shape[1] + 1  # calendar + log age

def get_window(user, eval_ts_s=None, end_local_idx=None):
    s, e = USER_BOUNDS[int(user)]
    t = EV_TIMES[s:e]

    if eval_ts_s is not None:
        right = int(np.searchsorted(t, int(eval_ts_s), side="right"))
    elif end_local_idx is not None:
        right = int(end_local_idx) + 1
    else:
        right = len(t)

    right = max(0, min(right, len(t)))
    left = max(0, right - MAX_EVENTS)

    vals = EV_VALUES[s+left:s+right]
    times = EV_TIMES[s+left:s+right]
    cal = EV_CAL[s+left:s+right]

    if len(times):
        anchor = int(eval_ts_s) if eval_ts_s is not None else int(times[-1])
        age_hours = np.maximum(anchor - times, 0).astype(np.float32) / 3600.0
        log_age = np.log1p(age_hours)[:, None].astype(np.float32)
        tf = np.concatenate([cal, log_age], axis=1)
    else:
        tf = np.zeros((0, TIME_DIM), dtype=np.float32)

    pad = MAX_EVENTS - len(vals)
    out_vals = np.full((MAX_EVENTS, N_FIELDS), TOK.pad_id, dtype=np.int32)
    out_tf = np.zeros((MAX_EVENTS, TIME_DIM), dtype=np.float32)
    out_pad = np.ones(MAX_EVENTS, dtype=bool)

    if len(vals):
        out_vals[pad:] = vals
        out_tf[pad:] = tf
        out_pad[pad:] = False

    return out_vals, out_tf, out_pad

def sample_pretrain_records(users, n_per_user, min_events=8, seed=SEED):
    rng = np.random.default_rng(seed)
    records = []
    for u in users:
        s, e = USER_BOUNDS.get(int(u), (0, 0))
        n = e - s
        if n < min_events:
            continue
        candidates = np.arange(min_events-1, n, dtype=np.int32)
        k = min(n_per_user, len(candidates))
        picks = rng.choice(candidates, size=k, replace=False)
        records.extend((int(u), int(j)) for j in picks)
    rng.shuffle(records)
    return records

PRETRAIN_RECORDS = sample_pretrain_records(
    SPLIT.train, CFG["windows_per_train_user"], CFG["min_pretrain_events"], SEED)
PRETRAIN_VAL_RECORDS = sample_pretrain_records(
    SPLIT.val, CFG["windows_per_val_user"], CFG["min_pretrain_events"], SEED+1)

print("pre-training records:", len(PRETRAIN_RECORDS))
print("pre-training validation records:", len(PRETRAIN_VAL_RECORDS))
print("event observations per pretrain pass (max):",
      len(PRETRAIN_RECORDS) * MAX_EVENTS)


In [ ]:
# 10. Dataset and dynamic masking

class PretrainDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        user, end_idx = self.records[idx]
        vals, tf, pad = get_window(user, end_local_idx=end_idx)
        prof = get_profile_values(user)
        return vals, tf, pad, prof

def collate_pretrain(batch):
    vals = torch.as_tensor(np.stack([x[0] for x in batch]), dtype=torch.long)
    tf = torch.as_tensor(np.stack([x[1] for x in batch]), dtype=torch.float32)
    pad = torch.as_tensor(np.stack([x[2] for x in batch]), dtype=torch.bool)
    prof = torch.as_tensor(np.stack([x[3] for x in batch]), dtype=torch.long)

    original = vals.clone()
    B, E, Fld = vals.shape
    valid = (~pad).unsqueeze(-1).expand(B, E, Fld)

    token_sel = (torch.rand(B, E, Fld) < CFG["token_mask_rate"]) & valid
    event_sel = (torch.rand(B, E, 1) < CFG["event_mask_rate"]).expand(B, E, Fld) & valid
    field_sel = (torch.rand(B, 1, Fld) < CFG["field_mask_rate"]).expand(B, E, Fld) & valid
    selected = token_sel | event_sel | field_sel

    if not selected.any():
        b = 0
        e = int((~pad[0]).nonzero()[0])
        selected[b, e, 0] = True

    unk_corrupt = selected & (torch.rand(B, E, Fld) < CFG["unk_corruption_rate"])
    loss_mask = selected & ~unk_corrupt

    vals[loss_mask] = TOK.mask_id
    vals[unk_corrupt] = TOK.unk_id

    return {
        "values": vals,
        "targets": original,
        "loss_mask": loss_mask,
        "time_features": tf,
        "event_pad": pad,
        "profile_values": prof,
    }

train_loader = DataLoader(
    PretrainDataset(PRETRAIN_RECORDS),
    batch_size=CFG["pretrain_batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_pretrain,
)

val_loader = DataLoader(
    PretrainDataset(PRETRAIN_VAL_RECORDS),
    batch_size=CFG["pretrain_batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_pretrain,
)

batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})
print("realised loss-mask rate:",
      float(batch["loss_mask"].sum() / (~batch["event_pad"]).sum() / N_FIELDS))


In [ ]:
# 11. Model definition

class MiniPragmaBackbone(nn.Module):
    def __init__(self, vocab_size, event_key_ids, profile_key_ids,
                 d_model=128, n_heads=4, d_ff=512,
                 profile_layers=1, event_layers=1, history_layers=2,
                 dropout=0.1, max_events=64, time_dim=7):
        super().__init__()
        self.d_model = d_model
        self.max_events = max_events
        self.n_event_fields = len(event_key_ids)
        self.n_profile_fields = len(profile_key_ids)

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=TOK.pad_id)
        self.register_buffer("event_key_ids",
                             torch.as_tensor(event_key_ids, dtype=torch.long))
        self.register_buffer("profile_key_ids",
                             torch.as_tensor(profile_key_ids, dtype=torch.long))

        self.event_field_pos = nn.Embedding(max(1, self.n_event_fields), d_model)
        self.profile_field_pos = nn.Embedding(max(1, self.n_profile_fields), d_model)
        self.history_pos = nn.Embedding(max_events + 1, d_model)

        def enc(n_layers):
            if n_layers <= 0:
                return nn.Identity()
            layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                dropout=dropout, activation="gelu",
                batch_first=True, norm_first=True)
            return nn.TransformerEncoder(layer, num_layers=n_layers)

        self.profile_encoder = enc(profile_layers)
        self.event_encoder = enc(event_layers)
        self.history_encoder = enc(history_layers)

        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.final_norm = nn.LayerNorm(d_model)

        # PRAGMA-style three-way MLM context.
        self.mlm_proj = nn.Sequential(
            nn.Linear(3 * d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )
        self.mlm_bias = nn.Parameter(torch.zeros(vocab_size))

        # Small BERT-style embedding initialisation.
        # Important because token_emb is also used as the tied MLM output matrix.
        nn.init.normal_(self.token_emb.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.event_field_pos.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.profile_field_pos.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.history_pos.weight, mean=0.0, std=0.02)

        # Preserve an exactly-zero padding embedding.
        with torch.no_grad():
            self.token_emb.weight[TOK.pad_id].zero_()

    def encode_profile(self, profile_values):
        B = profile_values.shape[0]
        usr = self.token_emb.weight[TOK.usr_id].view(1, 1, -1).expand(B, 1, -1)

        if self.n_profile_fields == 0:
            x = usr
        else:
            keys = self.profile_key_ids
            key_emb = self.token_emb(keys).view(1, self.n_profile_fields, -1)
            val_emb = self.token_emb(profile_values)
            pos = self.profile_field_pos(
                torch.arange(self.n_profile_fields, device=profile_values.device)
            ).view(1, self.n_profile_fields, -1)
            x = torch.cat([usr, key_emb + val_emb + pos], dim=1)

        x = self.profile_encoder(x)
        return x[:, 0]

    def encode_events(self, event_values):
        B, E, Fld = event_values.shape
        flat = event_values.reshape(B * E, Fld)

        evt = self.token_emb.weight[TOK.evt_id].view(1, 1, -1).expand(B * E, 1, -1)
        keys = self.token_emb(self.event_key_ids).view(1, Fld, -1)
        vals = self.token_emb(flat)
        pos = self.event_field_pos(
            torch.arange(Fld, device=event_values.device)
        ).view(1, Fld, -1)

        x = torch.cat([evt, keys + vals + pos], dim=1)
        x = self.event_encoder(x)

        evt_vec = x[:, 0].reshape(B, E, self.d_model)
        field_local = x[:, 1:].reshape(B, E, Fld, self.d_model)
        return evt_vec, field_local

    def encode(self, profile_values, event_values, time_features, event_pad):
        usr = self.encode_profile(profile_values)
        evt, field_local = self.encode_events(event_values)
        evt = evt + self.time_mlp(time_features)

        B, E, _ = evt.shape
        hist = torch.cat([usr[:, None, :], evt], dim=1)
        pos = self.history_pos(torch.arange(E + 1, device=evt.device))[None, :, :]
        hist = hist + pos

        hist_pad = torch.cat([
            torch.zeros(B, 1, dtype=torch.bool, device=evt.device),
            event_pad
        ], dim=1)

        if isinstance(self.history_encoder, nn.Identity):
            out = hist
        else:
            out = self.history_encoder(hist, src_key_padding_mask=hist_pad)

        out = self.final_norm(out)
        usr_ctx = out[:, 0]
        evt_ctx = out[:, 1:]
        return usr_ctx, evt_ctx, field_local

    def mlm_loss(self, profile_values, event_values, time_features,
                 event_pad, targets, loss_mask):
        usr_ctx, evt_ctx, field_local = self.encode(
            profile_values, event_values, time_features, event_pad)

        idx = loss_mask.nonzero(as_tuple=False)
        if len(idx) == 0:
            return torch.zeros((), device=event_values.device, requires_grad=True), None, None

        b, e, f = idx[:, 0], idx[:, 1], idx[:, 2]
        local = field_local[b, e, f]
        event = evt_ctx[b, e]
        user = usr_ctx[b]
        h = self.mlm_proj(torch.cat([local, event, user], dim=-1))

        logits = F.linear(h, self.token_emb.weight, self.mlm_bias)
        y = targets[b, e, f]
        loss = F.cross_entropy(logits, y)
        return loss, logits, idx

def build_backbone(seed=SEED_MODEL):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return MiniPragmaBackbone(
        vocab_size=TOK.vocab_size,
        event_key_ids=TOK.event_key_ids,
        profile_key_ids=TOK.profile_key_ids,
        d_model=CFG["d_model"],
        n_heads=CFG["n_heads"],
        d_ff=CFG["d_ff"],
        profile_layers=CFG["profile_layers"],
        event_layers=CFG["event_layers"],
        history_layers=CFG["history_layers"],
        dropout=CFG["dropout"],
        max_events=CFG["max_events"],
        time_dim=TIME_DIM,
    )

BACKBONE = build_backbone().to(DEVICE)

n_params = sum(p.numel() for p in BACKBONE.parameters())
print(f"Backbone parameters: {n_params:,} ({n_params/1e6:.3f}M)")

# Smoke forward pass.
b = {k: v.to(DEVICE) for k, v in batch.items()}
with torch.no_grad():
    loss, logits, idx = BACKBONE.mlm_loss(
        b["profile_values"], b["values"], b["time_features"],
        b["event_pad"], b["targets"], b["loss_mask"])
print("smoke MLM loss:", float(loss), "masked positions:", len(idx))


In [ ]:
# 12. Pre-training evaluation helpers: model vs unigram vs recency-copy

# Per-field training-user mode token.
# train_rows = TX["User"].isin(set(SPLIT.train.tolist())).to_numpy()
train_rows = np.isin(EV_USERS, SPLIT.train)
FIELD_MODE_IDS = []
for f in range(N_FIELDS):
    ids, cnt = np.unique(EV_VALUES[train_rows, f], return_counts=True)
    FIELD_MODE_IDS.append(int(ids[np.argmax(cnt)]))
FIELD_MODE_IDS = np.asarray(FIELD_MODE_IDS, dtype=np.int64)

@torch.no_grad()
def evaluate_mlm(model, loader, max_batches=None, seed=123):
    model.eval()
    torch.manual_seed(seed)

    losses, correct, total = [], 0, 0
    uni_correct, uni_total = 0, 0
    rec_correct, rec_total = 0, 0

    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        b = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        loss, logits, idx = model.mlm_loss(
            b["profile_values"], b["values"], b["time_features"],
            b["event_pad"], b["targets"], b["loss_mask"])
        losses.append(float(loss))
        if logits is None:
            continue

        pred = logits.argmax(dim=-1)
        y = b["targets"][idx[:,0], idx[:,1], idx[:,2]]
        correct += int((pred == y).sum())
        total += len(y)

        fidx = idx[:,2].cpu().numpy()
        uni = torch.as_tensor(FIELD_MODE_IDS[fidx], device=DEVICE)
        uni_correct += int((uni == y).sum())
        uni_total += len(y)

        # Same-field previous-event copy when a prior real event exists.
        prev_e = idx[:,1] - 1
        ok = prev_e >= 0
        if ok.any():
            bb = idx[ok,0]
            ee = prev_e[ok]
            ff = idx[ok,2]
            real_prev = ~b["event_pad"][bb, ee]
            if real_prev.any():
                bb, ee, ff = bb[real_prev], ee[real_prev], ff[real_prev]
                rec = b["targets"][bb, ee, ff]
                yy = b["targets"][idx[ok][real_prev,0],
                                  idx[ok][real_prev,1],
                                  idx[ok][real_prev,2]]
                rec_correct += int((rec == yy).sum())
                rec_total += len(yy)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": correct / max(total, 1),
        "unigram_accuracy": uni_correct / max(uni_total, 1),
        "recency_copy_accuracy": rec_correct / max(rec_total, 1),
        "n_masked": total,
    }

baseline_eval = evaluate_mlm(
    BACKBONE, val_loader, max_batches=min(10, CFG["validation_batches"]))
print("Random-init validation diagnostics:")
print(json.dumps(baseline_eval, indent=2))


In [ ]:
# 13. Pre-training loop

CHECKPOINT_PATH = CACHE_ROOT / "pretrained_backbone_public.pt"
TRAIN_HISTORY_PATH = CACHE_ROOT / "pretrain_history.json"

def lr_lambda(step):
    warm = CFG["warmup_steps"]
    total = CFG["max_pretrain_steps"]
    if step < warm:
        return max(step, 1) / max(warm, 1)
    progress = (step - warm) / max(total - warm, 1)
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

def train_pretrain(model, train_loader, val_loader, max_steps):
    model.train()
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["pretrain_lr"],
        weight_decay=CFG["pretrain_weight_decay"],
    )
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available()) if hasattr(torch, "amp") else torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    history = []
    step = 0
    t0 = time.time()
    iterator = iter(train_loader)

    while step < max_steps:
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            batch = next(iterator)

        b = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        opt.zero_grad(set_to_none=True)

        with (torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()) if hasattr(torch, "amp") else torch.cuda.amp.autocast(enabled=torch.cuda.is_available())):
            loss, _, _ = model.mlm_loss(
                b["profile_values"], b["values"], b["time_features"],
                b["event_pad"], b["targets"], b["loss_mask"])

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

        step += 1

        if step == 1 or step % 100 == 0:
            elapsed = time.time() - t0
            print(f"step {step:5d}/{max_steps}  loss {float(loss):.4f}  "
                  f"lr {sched.get_last_lr()[0]:.2e}  elapsed {elapsed/60:.1f}m")

        if step % CFG["validate_every"] == 0 or step == max_steps:
            metrics = evaluate_mlm(
                model, val_loader,
                max_batches=CFG["validation_batches"],
                seed=SEED + step)
            row = {"step": step, **metrics}
            history.append(row)
            print("VAL", row)
            model.train()

        if step % CFG["checkpoint_every"] == 0 or step == max_steps:
            torch.save({
                "step": step,
                "model_state": model.state_dict(),
                "cfg": CFG,
                "vocab_size": TOK.vocab_size,
                "event_fields": TOK.event_fields,
                "profile_fields": TOK.profile_fields,
            }, CHECKPOINT_PATH)
            with open(TRAIN_HISTORY_PATH, "w") as f:
                json.dump(history, f, indent=2)
            print("checkpoint ->", CHECKPOINT_PATH)

    return history

RUN_PRETRAIN = FORCE_REBUILD_CACHE or not CHECKPOINT_PATH.exists()

if CHECKPOINT_PATH.exists() and not RUN_PRETRAIN:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    BACKBONE.load_state_dict(ckpt["model_state"])
    print("Loaded existing checkpoint at step", ckpt["step"])
else:
    PRETRAIN_HISTORY = train_pretrain(
        BACKBONE, train_loader, val_loader, CFG["max_pretrain_steps"])


In [ ]:
# 7. Pretraining diagnostics — these figures are intended for the README.
if "PRETRAIN_HISTORY" not in globals():
    if TRAIN_HISTORY_PATH.exists():
        PRETRAIN_HISTORY = json.load(open(TRAIN_HISTORY_PATH))
    else:
        PRETRAIN_HISTORY = []

hist = pd.DataFrame(PRETRAIN_HISTORY)
if not hist.empty:
    display(hist)
    plt.figure(figsize=(8.6, 4.7))
    plt.plot(hist["step"], hist["accuracy"], marker="o", label="Transformer")
    plt.plot(hist["step"], hist["unigram_accuracy"], linestyle="--", label="Unigram")
    plt.plot(hist["step"], hist["recency_copy_accuracy"], linestyle="--", label="Recency copy")
    plt.xlabel("Pretraining step")
    plt.ylabel("Masked-token accuracy")
    plt.title("Masked self-supervised pretraining")
    plt.legend(frameon=False)
    savefig("07_pretraining_diagnostics.png")
    plt.show()

final_pretrain = evaluate_mlm(BACKBONE, val_loader, max_batches=CFG["validation_batches"], seed=SEED+999)
print(json.dumps(final_pretrain, indent=2))


# Part IV — Task 1: future category use

**Question:** will this customer use each of the five category groups in the next 30 days?

The headline comparison follows the conceptual ladder from the carousel:

1. **Persistence** — use previous-30-day category use as the forecast.
2. **Engineered history + GBDT** — aggregate 30/90-day history features.
3. **Scratch Transformer** — learn from the ordered history using only task labels.
4. **Pretrained Transformer + fine-tuning** — start from the self-supervised backbone, then adapt all weights to the task.

We also retain three representation ablations because they tell us *why* the neural results look the way they do: random frozen encoder, pretrained frozen encoder, and pretrained encoder without transaction history.


In [ ]:
# 15. Panel dataset backed by the same event store

Y_COLS = [f"y_{g}" for g in TARGET_GROUPS]
PREV_COLS = [f"prev_{g}" for g in TARGET_GROUPS]

class PanelSequenceDataset(Dataset):
    def __init__(self, panel, use_history=True):
        self.panel = panel.reset_index(drop=False).rename(columns={"index": "_row_id"})
        self.use_history = use_history

    def __len__(self):
        return len(self.panel)

    def __getitem__(self, i):
        r = self.panel.iloc[i]
        user = int(r["User"])
        eval_s = int(pd.Timestamp(r["eval_point"]).to_datetime64()
                     .astype("datetime64[s]").astype("int64"))

        if self.use_history:
            vals, tf, pad = get_window(user, eval_ts_s=eval_s)
        else:
            vals = np.full((MAX_EVENTS, N_FIELDS), TOK.pad_id, dtype=np.int32)
            tf = np.zeros((MAX_EVENTS, TIME_DIM), dtype=np.float32)
            pad = np.ones(MAX_EVENTS, dtype=bool)

        prof = get_profile_values(user)
        y = r[Y_COLS].to_numpy(dtype=np.float32)

        month = pd.Timestamp(r["eval_point"]).month
        eval_cal = np.array([
            math.sin(2*math.pi*(month-1)/12),
            math.cos(2*math.pi*(month-1)/12),
        ], dtype=np.float32)

        return vals, tf, pad, prof, y, user, int(r["_row_id"]), eval_cal

def collate_panel(batch):
    return {
        "values": torch.as_tensor(np.stack([x[0] for x in batch]), dtype=torch.long),
        "time_features": torch.as_tensor(np.stack([x[1] for x in batch]), dtype=torch.float32),
        "event_pad": torch.as_tensor(np.stack([x[2] for x in batch]), dtype=torch.bool),
        "profile_values": torch.as_tensor(np.stack([x[3] for x in batch]), dtype=torch.long),
        "labels": torch.as_tensor(np.stack([x[4] for x in batch]), dtype=torch.float32),
        "users": np.asarray([x[5] for x in batch], dtype=np.int32),
        "row_ids": np.asarray([x[6] for x in batch], dtype=np.int64),
        "eval_cal": torch.as_tensor(np.stack([x[7] for x in batch]), dtype=torch.float32),
    }

PANEL_DS = PanelSequenceDataset(PANEL, use_history=True)
print("panel sequence examples:", len(PANEL_DS))


In [ ]:
# 16. Evaluation metrics and exact panel split masks

TRAIN_USERS = set(SPLIT.train.tolist())
VAL_USERS = set(SPLIT.val.tolist())
TEST_USERS = set(SPLIT.test.tolist())

panel_user = PANEL["User"].to_numpy()
P_TR = np.isin(panel_user, list(TRAIN_USERS))
P_VA = np.isin(panel_user, list(VAL_USERS))
P_TE = np.isin(panel_user, list(TEST_USERS))

print("panel rows train/val/test:", P_TR.sum(), P_VA.sum(), P_TE.sum())
print("test users in panel:", PANEL.loc[P_TE, "User"].nunique())

def macro_map(y, score):
    aps = []
    for j in range(y.shape[1]):
        if y[:,j].min() == y[:,j].max():
            continue
        aps.append(average_precision_score(y[:,j], score[:,j]))
    return float(np.mean(aps)) if aps else np.nan

def per_group_ap(y, score):
    return {
        g: average_precision_score(y[:,j], score[:,j])
        for j, g in enumerate(TARGET_GROUPS)
        if y[:,j].min() != y[:,j].max()
    }

Y_ALL = PANEL[Y_COLS].to_numpy(np.int8)


In [ ]:
# 17. Cheap baselines: prior and persistence

train_prior = Y_ALL[P_TR].mean(axis=0)
S_PRIOR = np.tile(train_prior, (P_TE.sum(), 1))
S_PERSIST = PANEL.loc[P_TE, PREV_COLS].to_numpy(np.float32)
Y_TEST = Y_ALL[P_TE]

baseline_rows = [
    {"model": "Prior-only", "macro_mAP": macro_map(Y_TEST, S_PRIOR)},
    {"model": "Persistence (previous 30d)", "macro_mAP": macro_map(Y_TEST, S_PERSIST)},
]
display(pd.DataFrame(baseline_rows))
display(pd.DataFrame({
    "group": TARGET_GROUPS,
    "base_rate_test": Y_TEST.mean(axis=0),
    "prior_AP": list(per_group_ap(Y_TEST, S_PRIOR).values()),
    "persistence_AP": list(per_group_ap(Y_TEST, S_PERSIST).values()),
}))


In [ ]:
# 18. GBDT engineered-history baseline

# Main GBDT excludes snapshot profile variables.
# It uses only engineered historical fields plus evaluation month.
GBDT_FEATURES = [
    c for c in PANEL.columns
    if (
        c.startswith("p30_")
        or c.startswith("p90_")
        or c.startswith("prev_")
        or c == "eval_month"
    )
    and not c.startswith("y_")
]

# Keep only numeric / coercible fields.
X = PANEL[GBDT_FEATURES].copy()
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

GBDT_MODELS = []
S_GBDT = np.zeros((P_TE.sum(), len(TARGET_GROUPS)), dtype=np.float32)

for j, g in enumerate(TARGET_GROUPS):
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=SEED_MODEL + j,
        verbosity=-1,
    )
    model.fit(
        X.loc[P_TR], Y_ALL[P_TR, j],
        eval_set=[(X.loc[P_VA], Y_ALL[P_VA, j])],
        eval_metric="average_precision",
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)],
    )
    S_GBDT[:, j] = model.predict_proba(X.loc[P_TE])[:,1]
    GBDT_MODELS.append(model)

print("GBDT features:", len(GBDT_FEATURES))
print("GBDT macro mAP:", macro_map(Y_TEST, S_GBDT))
display(pd.DataFrame({"group": TARGET_GROUPS,
                      "AP": list(per_group_ap(Y_TEST, S_GBDT).values())}))


In [ ]:
# 8. Scratch and pretrained fine-tuned Transformers for Task 1.
class TaskModel(nn.Module):
    def __init__(self, backbone, n_outputs):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.d_model + 2, n_outputs)
    def forward(self, profile_values, event_values, time_features, event_pad, eval_cal):
        usr, _, _ = self.backbone.encode(profile_values, event_values, time_features, event_pad)
        return self.head(torch.cat([usr, eval_cal], dim=-1))


def panel_subset(panel, users):
    return panel[panel["User"].isin(set(users.tolist()))].copy().reset_index(drop=True)

TRAIN_PANEL_DS = PanelSequenceDataset(panel_subset(PANEL, SPLIT.train), use_history=True)
VAL_PANEL_DS = PanelSequenceDataset(panel_subset(PANEL, SPLIT.val), use_history=True)
TEST_PANEL_DS = PanelSequenceDataset(panel_subset(PANEL, SPLIT.test), use_history=True)


def task_loader(ds, shuffle, batch_size=None):
    return DataLoader(ds, batch_size=batch_size or CFG["downstream_batch_size"], shuffle=shuffle,
                      num_workers=0, pin_memory=torch.cuda.is_available(), collate_fn=collate_panel)

@torch.no_grad()
def predict_multilabel(model, ds):
    model.eval(); ys=[]; scores=[]; users=[]
    for b in task_loader(ds, False, 256):
        logits = model(b["profile_values"].to(DEVICE), b["values"].to(DEVICE),
                       b["time_features"].to(DEVICE), b["event_pad"].to(DEVICE), b["eval_cal"].to(DEVICE))
        ys.append(b["labels"].numpy()); scores.append(torch.sigmoid(logits).cpu().numpy()); users.append(b["users"])
    return np.concatenate(ys), np.concatenate(scores), np.concatenate(users)


def train_multilabel(model, train_ds, val_ds, epochs, lr):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=.01)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available()) if hasattr(torch, "amp") else torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_state, best_map = None, -np.inf
    history=[]
    for epoch in range(1, epochs+1):
        model.train(); losses=[]
        for b in task_loader(train_ds, True):
            opt.zero_grad(set_to_none=True)
            ctx = torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()) if hasattr(torch, "amp") else torch.cuda.amp.autocast(enabled=torch.cuda.is_available())
            with ctx:
                logits = model(b["profile_values"].to(DEVICE), b["values"].to(DEVICE), b["time_features"].to(DEVICE),
                               b["event_pad"].to(DEVICE), b["eval_cal"].to(DEVICE))
                loss = F.binary_cross_entropy_with_logits(logits, b["labels"].to(DEVICE))
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        yv, sv, _ = predict_multilabel(model, val_ds)
        val_map = macro_map(yv, sv); history.append({"epoch":epoch,"train_loss":np.mean(losses),"val_macro_mAP":val_map})
        print(f"epoch {epoch}: loss={np.mean(losses):.4f} val_mAP={val_map:.4f}")
        if val_map > best_map:
            best_map=val_map; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state); return model.to(DEVICE), pd.DataFrame(history)

SCRATCH_MODEL = TaskModel(build_backbone(seed=SEED_MODEL+99), len(TARGET_GROUPS))
SCRATCH_MODEL, SCRATCH_HIST = train_multilabel(SCRATCH_MODEL, TRAIN_PANEL_DS, VAL_PANEL_DS, CFG["scratch_epochs"], CFG["scratch_lr"])
Y_SCRATCH, S_SCRATCH, U_SCRATCH = predict_multilabel(SCRATCH_MODEL, TEST_PANEL_DS)
print("Scratch Transformer test macro mAP:", macro_map(Y_SCRATCH, S_SCRATCH))

FT_MODEL = TaskModel(copy.deepcopy(BACKBONE).cpu(), len(TARGET_GROUPS))
FT_MODEL, FT_HIST = train_multilabel(FT_MODEL, TRAIN_PANEL_DS, VAL_PANEL_DS, CFG["finetune_epochs"], CFG["finetune_lr"])
Y_FT, S_FT, U_FT = predict_multilabel(FT_MODEL, TEST_PANEL_DS)
print("Pretrained + fine-tuned test macro mAP:", macro_map(Y_FT, S_FT))


In [ ]:
# 9. Optional representation ablations.
if RUN_REPRESENTATION_ABLATIONS:
    # 19. Embedding extraction

    @torch.no_grad()
    def extract_embeddings(backbone, dataset, batch_size=256):
        loader = DataLoader(
            dataset, batch_size=batch_size, shuffle=False,
            num_workers=0, pin_memory=torch.cuda.is_available(),
            collate_fn=collate_panel)
        backbone.eval()

        Z, Y, U, R = [], [], [], []
        for batch in loader:
            vals = batch["values"].to(DEVICE, non_blocking=True)
            tf = batch["time_features"].to(DEVICE, non_blocking=True)
            pad = batch["event_pad"].to(DEVICE, non_blocking=True)
            prof = batch["profile_values"].to(DEVICE, non_blocking=True)
            eval_cal = batch["eval_cal"].to(DEVICE, non_blocking=True)

            usr, _, _ = backbone.encode(prof, vals, tf, pad)
            z = torch.cat([usr, eval_cal], dim=-1)

            Z.append(z.cpu().numpy())
            Y.append(batch["labels"].numpy())
            U.append(batch["users"])
            R.append(batch["row_ids"])

        return (
            np.concatenate(Z), np.concatenate(Y),
            np.concatenate(U), np.concatenate(R)
        )

    def fit_linear_probes(Z, Y, users):
        m_tr = np.isin(users, list(TRAIN_USERS))
        m_va = np.isin(users, list(VAL_USERS))
        m_te = np.isin(users, list(TEST_USERS))

        scores = np.zeros((m_te.sum(), Y.shape[1]), dtype=np.float32)
        models = []

        for j in range(Y.shape[1]):
            scaler = StandardScaler()
            Ztr = scaler.fit_transform(Z[m_tr])
            Zte = scaler.transform(Z[m_te])

            clf = LogisticRegression(
                max_iter=CFG["probe_max_iter"],
                solver="lbfgs",
                random_state=SEED_MODEL + j,
            )
            clf.fit(Ztr, Y[m_tr, j])
            scores[:,j] = clf.predict_proba(Zte)[:,1]
            models.append((scaler, clf))
        return scores, models

    # Pretrained embeddings
    Z_PRE, Y_PRE, U_PRE, R_PRE = extract_embeddings(BACKBONE, PANEL_DS)
    S_PRE_PROBE, PRE_PROBES = fit_linear_probes(Z_PRE, Y_PRE, U_PRE)
    print("Pretrained frozen probe macro mAP:",
          macro_map(Y_PRE[np.isin(U_PRE, list(TEST_USERS))], S_PRE_PROBE))

    # Random frozen control — same architecture, never trained.
    RANDOM_BACKBONE = build_backbone(seed=SEED_RANDOM_CONTROL).to(DEVICE)
    Z_RND, Y_RND, U_RND, R_RND = extract_embeddings(RANDOM_BACKBONE, PANEL_DS)
    S_RND_PROBE, RND_PROBES = fit_linear_probes(Z_RND, Y_RND, U_RND)
    print("Random frozen probe macro mAP:",
          macro_map(Y_RND[np.isin(U_RND, list(TEST_USERS))], S_RND_PROBE))
    # 20. History ablation on the pretrained frozen representation

    NOHIST_DS = PanelSequenceDataset(PANEL, use_history=False)
    Z_NOH, Y_NOH, U_NOH, R_NOH = extract_embeddings(BACKBONE, NOHIST_DS)
    S_NOH_PROBE, NOH_PROBES = fit_linear_probes(Z_NOH, Y_NOH, U_NOH)

    Y_NOH_TEST = Y_NOH[np.isin(U_NOH, list(TEST_USERS))]
    print("Pretrained probe WITH history :", macro_map(Y_TEST, S_PRE_PROBE))
    print("Pretrained probe WITHOUT history:", macro_map(Y_NOH_TEST, S_NOH_PROBE))
else:
    S_RND_PROBE = S_PRE_PROBE = S_NOH_PROBE = None


In [ ]:
# 10. Task 1 headline table, ablations and uncertainty.
def paired_user_bootstrap(y, score_a, score_b, users, metric_fn, n_boot=CFG["n_boot"], seed=SEED_BOOT):
    rng = np.random.default_rng(seed)
    uniq = np.unique(users)
    groups = {u: np.flatnonzero(users == u) for u in uniq}
    point = metric_fn(y, score_b) - metric_fn(y, score_a)
    gaps=[]
    for _ in range(n_boot):
        picked = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([groups[u] for u in picked])
        gaps.append(metric_fn(y[idx], score_b[idx]) - metric_fn(y[idx], score_a[idx]))
    lo,hi=np.nanpercentile(gaps,[2.5,97.5]); return float(point),float(lo),float(hi)

headline1 = pd.DataFrame([
    {"model":"Persistence", "macro_mAP":macro_map(Y_TEST,S_PERSIST)},
    {"model":"GBDT engineered history", "macro_mAP":macro_map(Y_TEST,S_GBDT)},
    {"model":"Scratch Transformer", "macro_mAP":macro_map(Y_TEST,S_SCRATCH)},
    {"model":"Pretrained + fine-tuned Transformer", "macro_mAP":macro_map(Y_TEST,S_FT)},
]).sort_values("macro_mAP",ascending=False).reset_index(drop=True)
display(headline1)
metric_bar(headline1, "model", "macro_mAP", "Task 1 — future category use", "08_task1_headline_benchmark.png", "Macro mAP")

per_cat=[]
for name,score in [("Persistence",S_PERSIST),("GBDT",S_GBDT),("Scratch Transformer",S_SCRATCH),("Pretrained + fine-tuned",S_FT)]:
    for g,ap in per_group_ap(Y_TEST,score).items(): per_cat.append({"model":name,"category":g,"AP":ap})
per_cat=pd.DataFrame(per_cat); display(per_cat)

point1,lo1,hi1=paired_user_bootstrap(Y_TEST,S_SCRATCH,S_FT,TEST_ENTITIES if 'TEST_ENTITIES' in globals() else PANEL.loc[P_TE,'User'].to_numpy(),macro_map)
print(f"Pretrained fine-tuned minus scratch: {point1:+.4f}; 95% user-bootstrap CI [{lo1:+.4f}, {hi1:+.4f}]")

if RUN_REPRESENTATION_ABLATIONS:
    ablations1=pd.DataFrame([
        {"model":"Random frozen + probe","macro_mAP":macro_map(Y_TEST,S_RND_PROBE)},
        {"model":"Pretrained no-history + probe","macro_mAP":macro_map(Y_TEST,S_NOH_PROBE)},
        {"model":"Pretrained frozen + probe","macro_mAP":macro_map(Y_TEST,S_PRE_PROBE)},
    ])
    display(ablations1)
    metric_bar(ablations1,"model","macro_mAP","Task 1 — representation ablations","09_task1_representation_ablations.png","Macro mAP")


# Part V — Task 2: future activity tier

The same pretrained backbone is now reused for a different target.

For each monthly evaluation point we count transactions in the next 30 days. LOW / MEDIUM / HIGH cut-points are estimated **only from training-user observations**, so validation and test targets do not influence the definition.

The four modelling strategies stay the same: recent-activity persistence, engineered-history GBDT, scratch Transformer, pretrained + fine-tuned Transformer.


In [ ]:
# 11. Build Task 2 labels with training-only thresholds.
future_count = PANEL["future_30d_tx_count"].to_numpy(np.int32)
q1, q2 = np.quantile(future_count[P_TR], [1/3, 2/3])

def to_tier(x):
    x=np.asarray(x); return np.where(x<=q1,0,np.where(x<=q2,1,2)).astype(np.int64)

Y2_ALL = to_tier(future_count)
Y2_TEST = Y2_ALL[P_TE]
prev_count = PANEL["p30_tx_count"].to_numpy(np.int32)
PREV_TIER = to_tier(prev_count)

activity_distribution=[]
for part,mask in [("train",P_TR),("validation",P_VA),("test",P_TE)]:
    vc=pd.Series(Y2_ALL[mask]).value_counts(normalize=True).reindex([0,1,2],fill_value=0)
    activity_distribution.append({"split":part,"LOW":vc[0],"MEDIUM":vc[1],"HIGH":vc[2]})
activity_distribution=pd.DataFrame(activity_distribution)
print(f"Training-only thresholds: q1={q1:.2f}, q2={q2:.2f}")
display(activity_distribution)


In [ ]:
# 12. Task 2 persistence and engineered-history GBDT.
train_freq=np.bincount(Y2_ALL[P_TR],minlength=3).astype(float); train_freq/=train_freq.sum()
S2_PRIOR=np.tile(train_freq,(P_TE.sum(),1))
prev_test=PREV_TIER[P_TE]
S2_RECENT=np.full((len(prev_test),3),.05,dtype=np.float32)
S2_RECENT[np.arange(len(prev_test)),prev_test]=.90; S2_RECENT/=S2_RECENT.sum(axis=1,keepdims=True)

GBDT_FEATURES2 = GBDT_FEATURES
X2 = PANEL[GBDT_FEATURES2].copy()
for c in X2.columns: X2[c]=pd.to_numeric(X2[c],errors="coerce")
X2=X2.replace([np.inf,-np.inf],np.nan).fillna(0).astype(np.float32)

model2=lgb.LGBMClassifier(objective="multiclass",num_class=3,n_estimators=500,learning_rate=.05,num_leaves=63,
                          min_child_samples=50,subsample=.8,subsample_freq=1,colsample_bytree=.8,reg_lambda=1.0,
                          random_state=SEED_MODEL+200,verbosity=-1)
model2.fit(X2.loc[P_TR],Y2_ALL[P_TR],eval_set=[(X2.loc[P_VA],Y2_ALL[P_VA])],
           callbacks=[lgb.early_stopping(40,verbose=False),lgb.log_evaluation(0)])
S2_GBDT=model2.predict_proba(X2.loc[P_TE])

def multiclass_macro_map(y_int, score, n_classes=3):
    y_oh=np.eye(n_classes,dtype=np.int8)[np.asarray(y_int,dtype=int)]
    return float(np.mean([average_precision_score(y_oh[:,j],score[:,j]) for j in range(n_classes)]))

print("Recent activity macro mAP:",multiclass_macro_map(Y2_TEST,S2_RECENT))
print("GBDT macro mAP:",multiclass_macro_map(Y2_TEST,S2_GBDT))


In [ ]:
# 13. Task 2 sequence datasets, scratch Transformer and pretrained fine-tuning.
class ActivitySequenceDataset(Dataset):
    def __init__(self, panel, labels):
        self.panel=panel.reset_index(drop=True); self.labels=np.asarray(labels,dtype=np.int64)
    def __len__(self): return len(self.panel)
    def __getitem__(self,i):
        r=self.panel.iloc[i]; user=int(r["User"])
        eval_s=int(pd.Timestamp(r["eval_point"]).to_datetime64().astype("datetime64[s]").astype("int64"))
        vals,tf,pad=get_window(user,eval_ts_s=eval_s); prof=get_profile_values(user)
        month=pd.Timestamp(r["eval_point"]).month
        eval_cal=np.array([math.sin(2*math.pi*(month-1)/12),math.cos(2*math.pi*(month-1)/12)],dtype=np.float32)
        return vals,tf,pad,prof,self.labels[i],user,eval_cal

def collate_activity(batch):
    return {
        "values":torch.as_tensor(np.stack([x[0] for x in batch]),dtype=torch.long),
        "time_features":torch.as_tensor(np.stack([x[1] for x in batch]),dtype=torch.float32),
        "event_pad":torch.as_tensor(np.stack([x[2] for x in batch]),dtype=torch.bool),
        "profile_values":torch.as_tensor(np.stack([x[3] for x in batch]),dtype=torch.long),
        "labels":torch.as_tensor(np.asarray([x[4] for x in batch]),dtype=torch.long),
        "users":np.asarray([x[5] for x in batch],dtype=np.int32),
        "eval_cal":torch.as_tensor(np.stack([x[6] for x in batch]),dtype=torch.float32),
    }

def activity_subset(mask):
    idx=np.flatnonzero(mask); return PANEL.iloc[idx].reset_index(drop=True),Y2_ALL[idx]
P2TR,Y2TR=activity_subset(P_TR); P2VA,Y2VA=activity_subset(P_VA); P2TE,Y2TE=activity_subset(P_TE)
T2_TRAIN=ActivitySequenceDataset(P2TR,Y2TR); T2_VAL=ActivitySequenceDataset(P2VA,Y2VA); T2_TEST=ActivitySequenceDataset(P2TE,Y2TE)

def loader2(ds,shuffle,batch_size=128):
    return DataLoader(ds,batch_size=batch_size,shuffle=shuffle,num_workers=0,pin_memory=torch.cuda.is_available(),collate_fn=collate_activity)

@torch.no_grad()
def predict2(model,ds):
    model.eval(); ys=[]; ss=[]; uu=[]
    for b in loader2(ds,False,256):
        logits=model(b["profile_values"].to(DEVICE),b["values"].to(DEVICE),b["time_features"].to(DEVICE),b["event_pad"].to(DEVICE),b["eval_cal"].to(DEVICE))
        ys.append(b["labels"].numpy()); ss.append(torch.softmax(logits,dim=-1).cpu().numpy()); uu.append(b["users"])
    return np.concatenate(ys),np.concatenate(ss),np.concatenate(uu)

def train2(model,epochs,lr):
    model=model.to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=.01)
    scaler=torch.amp.GradScaler("cuda",enabled=torch.cuda.is_available()) if hasattr(torch,"amp") else torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best=-np.inf; state=None; hist=[]
    for epoch in range(1,epochs+1):
        model.train(); losses=[]
        for b in loader2(T2_TRAIN,True,CFG["downstream_batch_size"]):
            opt.zero_grad(set_to_none=True)
            ctx=torch.amp.autocast(device_type="cuda",enabled=torch.cuda.is_available()) if hasattr(torch,"amp") else torch.cuda.amp.autocast(enabled=torch.cuda.is_available())
            with ctx:
                logits=model(b["profile_values"].to(DEVICE),b["values"].to(DEVICE),b["time_features"].to(DEVICE),b["event_pad"].to(DEVICE),b["eval_cal"].to(DEVICE))
                loss=F.cross_entropy(logits,b["labels"].to(DEVICE))
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); losses.append(float(loss.detach().cpu()))
        yv,sv,_=predict2(model,T2_VAL); m=multiclass_macro_map(yv,sv); hist.append({"epoch":epoch,"train_loss":np.mean(losses),"val_macro_mAP":m})
        print(f"epoch {epoch}: loss={np.mean(losses):.4f} val_mAP={m:.4f}")
        if m>best: best=m; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(state); return model.to(DEVICE),pd.DataFrame(hist)

T2_SCRATCH=TaskModel(build_backbone(seed=SEED_MODEL+299),3)
T2_SCRATCH,H2S=train2(T2_SCRATCH,CFG["scratch_epochs"],CFG["scratch_lr"])
Y2S,S2_SCRATCH,U2S=predict2(T2_SCRATCH,T2_TEST)

T2_FT=TaskModel(copy.deepcopy(BACKBONE).cpu(),3)
T2_FT,H2F=train2(T2_FT,CFG["finetune_epochs"],CFG["finetune_lr"])
Y2F,S2_FT,U2F=predict2(T2_FT,T2_TEST)

assert np.array_equal(Y2S,Y2F) and np.array_equal(U2S,U2F)
print("Scratch Transformer macro mAP:",multiclass_macro_map(Y2S,S2_SCRATCH))
print("Pretrained + fine-tuned macro mAP:",multiclass_macro_map(Y2F,S2_FT))


In [ ]:
# 14. Task 2 headline results and paired user bootstrap.
headline2=pd.DataFrame([
    {"model":"Recent-activity persistence","macro_mAP":multiclass_macro_map(Y2_TEST,S2_RECENT)},
    {"model":"GBDT engineered history","macro_mAP":multiclass_macro_map(Y2_TEST,S2_GBDT)},
    {"model":"Scratch Transformer","macro_mAP":multiclass_macro_map(Y2S,S2_SCRATCH)},
    {"model":"Pretrained + fine-tuned Transformer","macro_mAP":multiclass_macro_map(Y2F,S2_FT)},
]).sort_values("macro_mAP",ascending=False).reset_index(drop=True)
display(headline2)
metric_bar(headline2,"model","macro_mAP","Task 2 — future activity tier","10_task2_headline_benchmark.png","Macro mAP")

point2,lo2,hi2=paired_user_bootstrap(Y2F,S2_SCRATCH,S2_FT,U2F,multiclass_macro_map)
print(f"Pretrained fine-tuned minus scratch: {point2:+.4f}; 95% user-bootstrap CI [{lo2:+.4f}, {hi2:+.4f}]")


# Part VI — What does the full experiment say?

The final comparison should be read as three nested questions:

- **Does history help?** Compare recent-behaviour persistence with engineered historical features.
- **Does ordered sequence learning help?** Compare the engineered-history baseline with a Transformer trained from scratch.
- **Does self-supervised pretraining help?** Compare the same Transformer architecture starting from random weights versus the pretrained backbone.

The foundation-model case is therefore **not** “the Transformer must win every individual benchmark.” The strategic case becomes stronger when the learned representation can be reused across many tasks, labels are scarce, or repeated task-specific feature engineering becomes costly.


In [ ]:
# 15. Final two-task summary + README-ready figure and machine-readable tables.
summary = pd.concat([
    headline1.assign(task="Future category use"),
    headline2.assign(task="Future activity tier"),
], ignore_index=True)
summary["strategy"] = summary["model"].replace({
    "Persistence": "Persistence / recent behaviour",
    "Recent-activity persistence": "Persistence / recent behaviour",
})
display(summary[["task", "strategy", "macro_mAP"]])

pivot = summary.pivot(index="strategy", columns="task", values="macro_mAP").reindex([
    "Persistence / recent behaviour",
    "GBDT engineered history",
    "Scratch Transformer",
    "Pretrained + fine-tuned Transformer",
])
display(pivot)

fig, ax = plt.subplots(figsize=(10,5.2))
x=np.arange(len(pivot.index)); width=.36
cols=list(pivot.columns)
for j,c in enumerate(cols):
    ax.bar(x+(j-(len(cols)-1)/2)*width,pivot[c].to_numpy(),width,label=c)
ax.set_xticks(x); ax.set_xticklabels(pivot.index,rotation=20,ha="right")
ax.set_ylabel("Macro mAP"); ax.set_title("One behavioural backbone, two downstream tasks")
ax.legend(frameon=False)
savefig("11_final_two_task_comparison.png")
plt.show()

summary.to_csv(OUTPUT_ROOT/"headline_results.csv",index=False)
base_rates.to_csv(OUTPUT_ROOT/"task1_target_prevalence.csv",index=False)
activity_distribution.to_csv(OUTPUT_ROOT/"task2_target_distribution.csv",index=False)

run_manifest={
    "data_root":str(DATA_ROOT),
    "fast_mode":FAST_MODE,
    "config":CFG,
    "split_seed":SEED_SPLIT,
    "task1_pretrained_minus_scratch":{"point":point1,"ci_low":lo1,"ci_high":hi1},
    "task2_pretrained_minus_scratch":{"point":point2,"ci_low":lo2,"ci_high":hi2},
}
with open(OUTPUT_ROOT/"run_manifest.json","w") as f: json.dump(run_manifest,f,indent=2)
print("\nFinished. README-ready figures are in:",ARTIFACTS_DIR)
print("Headline results:",OUTPUT_ROOT/"headline_results.csv")


## Appendix — methodological notes

- **Synthetic data only.** Neither dataset contains real customer records.
- **Entity-disjoint evaluation.** Users/accounts are split before downstream evaluation so the same identity does not appear in train and test.
- **No fraud label in pretraining inputs.** The IBM fraud label is used only to preserve the original split stratification; it is not an event token.
- **Train-only Task 2 thresholds.** LOW / MEDIUM / HIGH activity cut-points are estimated from training users only.
- **Paired clustered uncertainty.** Pretrained-vs-scratch gaps are bootstrapped by user, not by individual panel row.
- **Public benchmark definition.** The raw IBM panel-building code in this notebook is the reproducible public definition. It should not be described as an exact reconstruction of the missing development-stage preprocessing.

When publishing results, commit the notebook plus the selected figures from `outputs/artifacts/` into the repository's `artifacts/` folder.
